In [4]:
!pip install snowflake

Defaulting to user installation because normal site-packages is not writeable

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.



  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 16.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 16.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   -------------- ------------------------- 4.5/12.1 MB 22.3 MB/s eta 0:00:01
   --------------------------- ------------ 8.4/12.1 MB 20.8 MB/s eta 0:00:01
   ---------------------------------------- 12.1/12.1 MB 22.2 MB/s eta 0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
   ---------------------------------------- 0.0/14.9 MB ? eta -:--:--
   ----------------- ---------------------- 6.6/14.9 MB 33.6 MB/s eta 0:00:01
   --------------------------------- ------ 12.6/14.9 MB 31.6 MB/s eta 0:00:01
   ---------------------------------------- 

In [12]:
!pip install dotenv

Defaulting to user installation because normal site-packages is not writeable


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
from dotenv import load_dotenv
import os
import requests
import snowflake.connector
import json
from datetime import datetime

# Fetch data from the SDG API
# DONE: url = "https://unstats.un.org/SDGAPI/v1/sdg/GeoArea/List"
# DONE: url = "https://unstats.un.org/SDGAPI/v1/sdg/GeoArea/Tree"

url = "https://unstats.un.org/SDGAPI/v1/sdg/GeoArea/Tree"
response = requests.get(url)

data = response.json()

# print(data[:2])

load_dotenv()

conn = snowflake.connector.connect(
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=os.getenv("SNOWFLAKE_DATABASE"),
    schema=os.getenv("SNOWFLAKE_SCHEMA"),
    role=os.getenv("SNOWFLAKE_ROLE")
)

cur = conn.cursor()

for row in data:
    cur.execute("""
    INSERT INTO GEO_AREA_RAW(endpoint, load_time, payload)
    SELECT %s, %s, PARSE_JSON(%s)
    """,
    ("GeoArea/Tree", datetime.utcnow(), json.dumps(row))
    )

cur.close()
conn.close()

C:\Users\kmddg\AppData\Local\Temp\ipykernel_18600\1475101884.py:37: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ("GeoArea/Tree", datetime.utcnow(), json.dumps(row))
